# Generate Redesigned Fig. 2: Three GEDI Height-Class Histograms Side-by-Side

This notebook is autonomous: it reads the verified GEDI distribution table generated by `Study_Area.ipynb` and does not depend on another live kernel.

This notebook generates the redesigned Fig. 2 with three histograms side-by-side (no spatial maps)
- Output: `Fig_02_GEDI_Height_Class_Distribution_Three_Forests.{png,svg,pdf}`
- Source table: `Results/Final_Article/Study_Area/GEDI_RH95_unique_shot_distribution_by_split.csv`
- Output: `Results/Final_Article/Study_Area/Fig_02_GEDI_Height_Class_Distribution_Three_Forests.{png,svg,pdf}`

> This is an alternative histogram-only figure. It does not replace the official split-map figure unless the manuscript is explicitly changed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
OUT_DIR = PROJECT / "Results" / "Final_Article" / "Study_Area"
OUT_DIR.mkdir(parents=True, exist_ok=True)
GEDI_TABLE_PATH = OUT_DIR / "GEDI_RH95_unique_shot_distribution_by_split.csv"
EXPORT_DPI = 600
SPLIT_COLORS = {"train": "#7AAED1", "val": "#E77B70", "test": "#F2F2F2"}
SPLIT_LABEL = {"train": "Train", "val": "Val", "test": "Test"}
FORESTS = ["Ifran", "Maamoura", "Agadir"]

if not GEDI_TABLE_PATH.is_file():
    raise FileNotFoundError(
        f"Missing verified GEDI distribution table: {GEDI_TABLE_PATH}\n"
        "Run Study_Area.ipynb once to regenerate the source table."
    )
GEDI_DISTRIBUTIONS = pd.read_csv(GEDI_TABLE_PATH)
required = {"forest", "split", "height_class", "count", "split_n"}
missing = required - set(GEDI_DISTRIBUTIONS.columns)
if missing:
    raise RuntimeError(f"Invalid GEDI distribution table; missing columns: {sorted(missing)}")
GEDI_DISTRIBUTIONS["split"] = GEDI_DISTRIBUTIONS["split"].replace({"validation": "val", "valid": "val"})
if set(GEDI_DISTRIBUTIONS["forest"]) != set(FORESTS):
    raise RuntimeError(f"Unexpected forests: {sorted(GEDI_DISTRIBUTIONS['forest'].unique())}")
print(f"[READY] Source: {GEDI_TABLE_PATH}")
print(f"[READY] Rows: {len(GEDI_DISTRIBUTIONS):,}; output: {OUT_DIR}")

## Define the histogram plotting function

In [ ]:
def _height_class_lower_bound(label):
    text = str(label).replace("–", "-").replace("m", "").strip()
    return float(text.split("-", 1)[0].strip())


def plot_gedi_histogram_compact(ax, forest, gedi_table, ylabel):
    """Plot one compact GEDI height-class histogram."""
    # Filter to this forest only
    table = gedi_table[gedi_table["forest"].eq(forest)]

    # Extract height class labels in order
    labels = sorted(table["height_class"].dropna().astype(str).unique(), key=_height_class_lower_bound)
    x = np.arange(len(labels), dtype=float)
    width = 0.25

    for offset, split in zip((-1, 0, 1), ("train", "val", "test")):
        part = table[table["split"].eq(split)].set_index("height_class").reindex(labels)
        # Test split: light gray with diagonal hatching
        is_test = split == "test"
        ax.bar(
            x + offset * width, part["count"].to_numpy(dtype=np.float32),
            width=width,
            color="#F2F2F2" if is_test else SPLIT_COLORS.get(split, "#7AAED1"),
            edgecolor="black" if is_test else "none",
            linewidth=1.0 if is_test else 0.0,
            hatch="////" if is_test else None,
            alpha=1.0 if is_test else 0.90,
            zorder=3 if is_test else 2,
            label=("Train" if split == "train" else
                   "Val" if split == "val" else "Test"),
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
    ecosystem = {
        "Ifran": "Moderately dense",
        "Maamoura": "Low density",
        "Agadir": "Sparse",
    }.get(forest, forest)
    ax.set_title(f"{ecosystem} — {forest} Forest", fontweight="semibold", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlabel("GEDI RH95 height class", fontsize=9)
    ax.grid(axis="y", alpha=0.22, zorder=1)
    ax.set_axisbelow(True)

print("✓ Function defined: plot_gedi_histogram_compact()")

## Create and save the three-column histogram layout

In [ ]:
# Create the 3-column histogram layout
fig, axes = plt.subplots(1, 3, figsize=(15.0, 5.0), constrained_layout=True)

for ax, forest in zip(axes, ["Ifran", "Maamoura", "Agadir"]):
    plot_gedi_histogram_compact(
        ax, forest, GEDI_DISTRIBUTIONS,
        ylabel="Unique GEDI RH95 shots"
    )

# Unified legend at the bottom
handles = [
    Patch(facecolor=SPLIT_COLORS.get(s, "#7AAED1"),
          edgecolor=("black" if s == "test" else "none"),
          hatch=("////" if s == "test" else None),
          alpha=(1.0 if s == "test" else 0.90),
          label=SPLIT_LABEL.get(s, s.capitalize()))
    for s in ("train", "val", "test")
]
fig.legend(
    handles=handles,
    loc="lower center",
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, -0.08),
    columnspacing=1.5,
)

plt.show()
print("✓ Figure created!")

## Save as main Fig. 2 (PNG, SVG, PDF)

In [ ]:
# Save as main Fig. 2
fig_2_paths = {
    "png": OUT_DIR / "Fig_02_GEDI_Height_Class_Distribution_Three_Forests.png",
    "svg": OUT_DIR / "Fig_02_GEDI_Height_Class_Distribution_Three_Forests.svg",
    "pdf": OUT_DIR / "Fig_02_GEDI_Height_Class_Distribution_Three_Forests.pdf",
}

for fmt, path in fig_2_paths.items():
    if fmt == "png":
        fig.savefig(path, dpi=EXPORT_DPI, bbox_inches="tight", facecolor="white")
    else:
        fig.savefig(path, bbox_inches="tight", facecolor="white")

plt.close(fig)

print("\n" + "="*70)
print("✓ REDESIGNED FIG. 2 SAVED SUCCESSFULLY!")
print("="*70)
for fmt, path in fig_2_paths.items():
    print(f"  {fmt.upper()}: {path}")

## Summary: Page Reduction

In [ ]:
print("\n" + "="*70)
print("PAGE REDUCTION SUMMARY")
print("="*70)
print("\nMain text Fig. 2 change:")
print("  BEFORE: 3 spatial maps + 3 histograms (1.5 pages)")
print("  AFTER:  3 histograms side-by-side (0.4 pages)")
print("  SAVINGS: ~1.1 pages ✓")
print("\nAppendix gain:")
print("  BEFORE: Fig. 2 in main text + separate spatial maps somewhere")
print("  AFTER:  Appendix Fig. A1 (spatial splits) + clear reference")
print("  RESULT: Cleaner narrative flow ✓")
print("\n" + "="*70)
print("✓ Next step: Use the new Fig_02 files in your manuscript!")
print("="*70)